In [0]:
%pip install rasterio

#/dbfs/mnt/base/unrestricted/source_environment_agency/dataset_national_lidar_programme_dtm/format_GEOTIFF_national_lidar_programme_dtm/LATEST_national_lidar_programme_dtm/DTM/TL78nw_DTM_1m.tif



In [0]:
import os
import glob
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import Affine
from rasterio.merge import merge
from rasterio.io import MemoryFile
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Settings
# --------------------------------------------------

input_folder = r"/dbfs/FileStore/AndyBeverton/LIDAR_selected"

output_folder = r"/dbfs/FileStore/AndyBeverton/LIDAR_Merged_10m"
os.makedirs(output_folder, exist_ok=True)

output_tif = os.path.join(output_folder, "merged_lidar_DTM_10m.tif")

target_resolution = 10

test_mode = False      # start with True
test_file_count = 3   # test first 3 tiles
safe_nodata = -9999.0

# --------------------------------------------------
# 2. Find input TIFFs
# --------------------------------------------------

input_tifs = sorted(glob.glob(os.path.join(input_folder, "*.tif")))

if test_mode:
    input_tifs = input_tifs[:test_file_count]

print(f"Input TIFFs to process: {len(input_tifs)}")

for tif in input_tifs:
    print(os.path.basename(tif))

if len(input_tifs) == 0:
    raise ValueError("No TIFFs found.")


# --------------------------------------------------
# 3. Resample to 10 m in memory
# --------------------------------------------------

memory_files = []
datasets = []
failed = []

for i, tif in enumerate(input_tifs, start=1):
    print(f"[{i}/{len(input_tifs)}] Reading and resampling {os.path.basename(tif)}")

    try:
        with rasterio.open(tif) as src:
            xres, yres = src.res

            scale_x = target_resolution / xres
            scale_y = target_resolution / abs(yres)

            new_width = int(src.width / scale_x)
            new_height = int(src.height / scale_y)

            data = src.read(
                1,
                out_shape=(new_height, new_width),
                resampling=Resampling.average
            )

            new_transform = src.transform * Affine.scale(
                src.width / new_width,
                src.height / new_height
            )

            profile = src.profile.copy()
            profile.update(
                height=new_height,
                width=new_width,
                transform=new_transform,
                compress="lzw",
                tiled=False,
                BIGTIFF="IF_SAFER"
            )

        memfile = MemoryFile()

        with memfile.open(**profile) as mem:
            mem.write(data, 1)

        dataset = memfile.open()

        memory_files.append(memfile)
        datasets.append(dataset)

        print("   OK:", dataset.res, dataset.width, "x", dataset.height)

    except Exception as e:
        print("   FAILED:", e)
        failed.append(tif)

print("Successful in-memory rasters:", len(datasets))
print("Failed:", len(failed))

if len(datasets) == 0:
    raise ValueError("No valid rasters available to merge.")


# --------------------------------------------------
# 4. Merge in-memory 10 m rasters
# --------------------------------------------------

print("Merging in-memory 10 m rasters...")

mosaic, out_transform = merge(
    datasets,
    nodata=safe_nodata
)
out_meta = datasets[0].meta.copy()
out_meta.update(
    driver="GTiff",
    height=mosaic.shape[1],
    width=mosaic.shape[2],
    transform=out_transform,
    compress="lzw",
    tiled=False,
    BIGTIFF="IF_SAFER"
)

print("Merged shape:", mosaic.shape)

# --------------------------------------------------
# 5. Save final merged TIFF via MemoryFile
# --------------------------------------------------

from rasterio.io import MemoryFile

output_tif = r"/dbfs/FileStore/AndyBeverton/LIDAR_Merged_10m/merged_lidar_DTM_10m.tif"

# Use a safer nodata value
out_meta.update(
    driver="GTiff",
    height=mosaic.shape[1],
    width=mosaic.shape[2],
    transform=out_transform,
    dtype="float32",
    nodata=-9999.0,
    compress="lzw",
    tiled=False,
    BIGTIFF="IF_SAFER"
)

# Replace extreme nodata values with -9999
mosaic = mosaic.astype("float32")
mosaic[np.isclose(mosaic, -3.4028235e38)] = -9999.0

print("Writing final GeoTIFF to memory first...")

with MemoryFile() as memfile:
    with memfile.open(**out_meta) as dst:
        dst.write(mosaic)

    print("Writing completed GeoTIFF bytes to DBFS...")
    with open(output_tif, "wb") as f:
        f.write(memfile.read())

print("Saved:")
print(output_tif)


# --------------------------------------------------
# 6. Plot directly from in-memory mosaic
# --------------------------------------------------
lidar = mosaic[0].astype("float32")
lidar = np.where(lidar == safe_nodata, np.nan, lidar)

print("Merged min:", np.nanmin(lidar))
print("Merged max:", np.nanmax(lidar))
print("Merged mean:", np.nanmean(lidar))

vmin = np.nanpercentile(lidar, 2)
vmax = np.nanpercentile(lidar, 98)

plt.figure(figsize=(10, 8))
plt.imshow(
    lidar,
    cmap="terrain",
    extent=[left, right, bottom, top],
    origin="upper",
    vmin=vmin,
    vmax=vmax
)

plt.colorbar(label="Elevation / height, m")
plt.xlabel("Easting")
plt.ylabel("Northing")
plt.title("Merged LiDAR DTM, 10 m")
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 7. Optional: validate saved file
# --------------------------------------------------

try:
    with rasterio.open(output_tif) as src:
        print("Saved file opens successfully.")
        print("CRS:", src.crs)
        print("Resolution:", src.res)
        print("Size:", src.width, "x", src.height)
        print("Bounds:", src.bounds)
except Exception as e:
    print("Saved file could not be reopened from DBFS.")
    print("But the in-memory merge plotted successfully.")
    print("Error:", e)


# --------------------------------------------------
# 8. Clean up open in-memory files
# --------------------------------------------------

for ds in datasets:
    ds.close()

for mf in memory_files:
    mf.close()

In [0]:
import os
import glob
import math
import geopandas as gpd

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

lidar_folder = r"/dbfs/mnt/base/unrestricted/source_environment_agency/dataset_national_lidar_programme_dtm/format_GEOTIFF_national_lidar_programme_dtm/LATEST_national_lidar_programme_dtm/DTM"

shapefile_path = r"/dbfs/FileStore/AndyBeverton/33035/33035.shp"

output_folder_dbfs = "FileStore/AndyBeverton/LIDAR_selected"
output_folder_path = "/dbfs/" + output_folder_dbfs

dbutils.fs.mkdirs(output_folder_dbfs)

# --------------------------------------------------
# 2. Read shapefile and get extent
# --------------------------------------------------

shape_gdf = gpd.read_file(shapefile_path)

if shape_gdf.crs is None:
    raise ValueError("Shapefile has no CRS.")

if shape_gdf.crs.to_epsg() != 27700:
    shape_gdf = shape_gdf.to_crs("EPSG:27700")

xmin, ymin, xmax, ymax = shape_gdf.total_bounds

print("Shapefile extent:")
print("West:", xmin)
print("South:", ymin)
print("East:", xmax)
print("North:", ymax)

# --------------------------------------------------
# 3. OS National Grid tile helpers
# --------------------------------------------------

def get_100km_letters(easting, northing):
    letters = "ABCDEFGHJKLMNOPQRSTUVWXYZ"  # no I

    e100 = int(easting // 100000)
    n100 = int(northing // 100000)

    if not (0 <= e100 <= 6 and 0 <= n100 <= 12):
        return None

    l1 = (19 - n100) - ((19 - n100) % 5) + ((e100 + 10) // 5)
    l2 = ((19 - n100) * 5 % 25) + (e100 % 5)

    return letters[l1] + letters[l2]


def get_5km_tile_name(easting, northing):
    letters = get_100km_letters(easting, northing)

    if letters is None:
        return None

    local_e = int(easting % 100000)
    local_n = int(northing % 100000)

    ten_km_e = local_e // 10000
    ten_km_n = local_n // 10000

    within_10km_e = local_e % 10000
    within_10km_n = local_n % 10000

    east_west = "e" if within_10km_e >= 5000 else "w"
    north_south = "n" if within_10km_n >= 5000 else "s"

    return f"{letters}{ten_km_e}{ten_km_n}{north_south}{east_west}"


# --------------------------------------------------
# 4. Generate tile names from shapefile extent
# --------------------------------------------------

tile_size = 5000

x_start = math.floor(xmin / tile_size) * tile_size
x_end = math.floor(xmax / tile_size) * tile_size

y_start = math.floor(ymin / tile_size) * tile_size
y_end = math.floor(ymax / tile_size) * tile_size

required_tile_names = []

for x in range(int(x_start), int(x_end) + tile_size, tile_size):
    for y in range(int(y_start), int(y_end) + tile_size, tile_size):
        tile_name = get_5km_tile_name(x + 1, y + 1)
        if tile_name is not None:
            required_tile_names.append(tile_name)

required_tile_names = sorted(set(required_tile_names))

print(f"Tiles required from extent: {len(required_tile_names)}")
print(required_tile_names[:20], "...")

# --------------------------------------------------
# 5. Find matching source TIFFs by name only
# --------------------------------------------------

source_tifs = []

for tile_name in required_tile_names:
    tif_path = os.path.join(lidar_folder, f"{tile_name}_DTM_1m.tif")

    if os.path.exists(tif_path):
        source_tifs.append(tif_path)

print(f"Existing source TIFFs found: {len(source_tifs)}")

# --------------------------------------------------
# 6. Copy selected TIFFs to output folder
# --------------------------------------------------

for i, src_path in enumerate(source_tifs, start=1):
    file_name = os.path.basename(src_path)
    dst_dbfs = f"{output_folder_dbfs}/{file_name}"

    print(f"[{i}/{len(source_tifs)}] Copying {file_name}")

    # Skip if already copied
    if os.path.exists(os.path.join(output_folder_path, file_name)):
        print("   already exists, skipping")
        continue

    dbutils.fs.cp(
        src_path.replace("/dbfs/", "dbfs:/"),
        dst_dbfs
    )

print("Done.")
print("Copied files to:")
print(output_folder_path)

In [0]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from rasterio.plot import show

# --------------------------------------------------
# 1. Set path to your LiDAR .tif file
# --------------------------------------------------
lidar_tif = r"/dbfs/FileStore/AndyBeverton/LIDAR_Merged_10m/merged_lidar_DTM_10m.tif"#/dbfs/mnt/base/unrestricted/source_environment_agency/dataset_national_lidar_programme_dtm/format_GEOTIFF_national_lidar_programme_dtm/LATEST_national_lidar_programme_dtm/DTM/TL78nw_DTM_1m.tif"

# --------------------------------------------------
# 2. Open raster
# --------------------------------------------------
with rasterio.open(lidar_tif) as src:
    lidar = src.read(1)  # read first band
    profile = src.profile
    bounds = src.bounds
    crs = src.crs
    nodata = src.nodata

# --------------------------------------------------
# 3. Replace NoData values with NaN
# --------------------------------------------------
if nodata is not None:
    lidar = np.where(lidar == nodata, np.nan, lidar)

# --------------------------------------------------
# 4. Print raster information
# --------------------------------------------------
print("CRS:", crs)
print("Bounds:", bounds)
print("NoData value:", nodata)
print("Min elevation:", np.nanmin(lidar))
print("Max elevation:", np.nanmax(lidar))

# --------------------------------------------------
# 5. Plot LiDAR raster
# --------------------------------------------------
plt.figure(figsize=(10, 8))

plt.imshow(
    lidar,
    cmap="terrain",
    extent=[
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ]
)

plt.colorbar(label="Elevation / height")
plt.xlabel("Easting")
plt.ylabel("Northing")
plt.title("LiDAR raster")
plt.tight_layout()
plt.show()

In [0]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
# --------------------------------------------------
# 1. Set paths
# --------------------------------------------------

lidar_tif = r"/dbfs/FileStore/AndyBeverton/LIDAR_Merged_10m/merged_lidar_DTM_10m.tif"

polygon_shp_1 = r"/dbfs/FileStore/AndyBeverton/33035/33035.shp"

# Leave as None if not using
polygon_shp_2 = None
point_shp_1 = None

shapefiles = {}

if polygon_shp_1:
    shapefiles["Polygon 1"] = {
        "path": polygon_shp_1,
        "type": "polygon",
        "colour": "black",
        "linewidth": 2,
        "markersize": None
    }

if polygon_shp_2:
    shapefiles["Polygon 2"] = {
        "path": polygon_shp_2,
        "type": "polygon",
        "colour": "red",
        "linewidth": 2,
        "markersize": None
    }

if point_shp_1:
    shapefiles["Points"] = {
        "path": point_shp_1,
        "type": "point",
        "colour": "blue",
        "linewidth": None,
        "markersize": 30
    }

# --------------------------------------------------
# 2. Plot options
# --------------------------------------------------

zoom_to_shapefiles = True
buffer_m = 1000  # plot 1000 m beyond shapefile extent

# --------------------------------------------------
# 3. Open raster
# --------------------------------------------------

with rasterio.open(lidar_tif) as src:
    lidar = src.read(1)
    bounds = src.bounds
    crs = src.crs
    nodata = src.nodata

if nodata is not None:
    lidar = np.where(lidar == nodata, np.nan, lidar)

print("CRS:", crs)
print("Bounds:", bounds)
print("NoData value:", nodata)
print("Min elevation:", np.nanmin(lidar))
print("Max elevation:", np.nanmax(lidar))

# --------------------------------------------------
# 4. Read shapefiles and reproject to raster CRS
# --------------------------------------------------

shape_layers = {}

for name, info in shapefiles.items():
    gdf = gpd.read_file(info["path"])

    if gdf.crs is None:
        raise ValueError(f"{name} has no CRS.")

    if gdf.crs != crs:
        gdf = gdf.to_crs(crs)

    shape_layers[name] = gdf

    print(f"{name} bounds:", gdf.total_bounds)


# --------------------------------------------------
# 5. Work out plot extent
# --------------------------------------------------

if zoom_to_shapefiles and len(shape_layers) > 0:

    all_bounds = np.array([
        gdf.total_bounds for gdf in shape_layers.values()
    ])

    xmin = all_bounds[:, 0].min() - buffer_m
    ymin = all_bounds[:, 1].min() - buffer_m
    xmax = all_bounds[:, 2].max() + buffer_m
    ymax = all_bounds[:, 3].max() + buffer_m

else:
    xmin = bounds.left
    ymin = bounds.bottom
    xmax = bounds.right
    ymax = bounds.top

# --------------------------------------------------
# 6. Plot LiDAR raster
# --------------------------------------------------

plt.figure(figsize=(12, 10))
ax = plt.gca()

img = ax.imshow(
    lidar,
    cmap="terrain",
    extent=[
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ],
    origin="upper"
)

# --------------------------------------------------
# 7. Plot shapefiles over the top
# --------------------------------------------------

for name, info in shapefiles.items():
    gdf = shape_layers[name]

    if info["type"] == "polygon":
        gdf.boundary.plot(
            ax=ax,
            color=info["colour"],
            linewidth=info["linewidth"],
            label=name
        )

    elif info["type"] == "point":
        gdf.plot(
            ax=ax,
            color=info["colour"],
            markersize=info["markersize"],
            label=name
        )

# --------------------------------------------------
# 8. Finish plot
# --------------------------------------------------

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

plt.colorbar(img, ax=ax, label="Elevation / height")
ax.set_xlabel("Easting")
ax.set_ylabel("Northing")
ax.set_title("LiDAR raster with shapefile overlays")

if len(shape_layers) > 0:
    ax.legend(
        loc="upper right",
        frameon=True
    )


plt.tight_layout()
plt.show()

In [0]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
from rasterio.features import geometry_mask

# --------------------------------------------------
# 1. Set paths
# --------------------------------------------------

lidar_tif = r"/dbfs/FileStore/AndyBeverton/LIDAR_Merged_10m/merged_lidar_DTM_10m.tif"

polygon_shp_1 = r"/dbfs/FileStore/AndyBeverton/33035_minus_contributingcatchments_out/33035_minus_contributingcatchments.shp"#/dbfs/FileStore/AndyBeverton/33035/33035.shp"

# Leave as None if not using
gauge_point_shp = r"/dbfs/FileStore/tables/AndyBeverton/Point Data/Denver_stations.shp"
point_shp_1 = r"/dbfs/FileStore/tables/AndyBeverton/Point Data/Denver_Complex.shp"

shapefiles = {}

if polygon_shp_1:
    shapefiles["Area of Investigation"] = {
        "path": polygon_shp_1,
        "type": "polygon",
        "colour": "black",
        "linewidth": 1,
        "markersize": None
    }

if gauge_point_shp:
    shapefiles["Gauge Locations"] = {
        "path": gauge_point_shp,
        "type": "point",
        "colour": "lime",
        "linewidth": None,
        "markersize": 90
    }

if point_shp_1:
    shapefiles["Denver Complex"] = {
        "path": point_shp_1,
        "type": "point",
        "colour": "red",
        "linewidth": None,
        "markersize": 90
    }

# --------------------------------------------------
# 2. Plot options
# --------------------------------------------------

zoom_to_shapefiles = True
buffer_m = 0  # plot 1000 m beyond shapefile extent

# Only plot LiDAR inside a chosen shapefile?
plot_lidar_inside_shapefile_only = True

# Choose which shapefile to use as the mask
# Must match one of the keys in shapefiles, e.g. "Polygon 1"
mask_shapefile_name = "Area of Investigation"

# --------------------------------------------------
# 3. Open raster
# --------------------------------------------------

with rasterio.open(lidar_tif) as src:
    lidar = src.read(1)
    bounds = src.bounds
    crs = src.crs
    nodata = src.nodata
    transform = src.transform
    raster_shape = src.shape

if nodata is not None:
    lidar = np.where(lidar == nodata, np.nan, lidar)

print("CRS:", crs)
print("Bounds:", bounds)
print("NoData value:", nodata)
print("Min elevation:", np.nanmin(lidar))
print("Max elevation:", np.nanmax(lidar))

# --------------------------------------------------
# 4. Read shapefiles and reproject to raster CRS
# --------------------------------------------------

shape_layers = {}

for name, info in shapefiles.items():
    gdf = gpd.read_file(info["path"])

    if gdf.crs is None:
        raise ValueError(f"{name} has no CRS.")

    if gdf.crs != crs:
        gdf = gdf.to_crs(crs)

    shape_layers[name] = gdf

    print(f"{name} bounds:", gdf.total_bounds)

# --------------------------------------------------
# 5. Optionally mask LiDAR to one shapefile
# --------------------------------------------------

lidar_to_plot = lidar.copy()

if plot_lidar_inside_shapefile_only:

    if mask_shapefile_name not in shape_layers:
        raise ValueError(
            f"mask_shapefile_name '{mask_shapefile_name}' not found. "
            f"Available shapefiles are: {list(shape_layers.keys())}"
        )

    mask_gdf = shape_layers[mask_shapefile_name]

    # Keep only valid polygon geometries
    mask_geoms = [
        geom for geom in mask_gdf.geometry
        if geom is not None and not geom.is_empty
    ]

    if len(mask_geoms) == 0:
        raise ValueError(f"No valid geometries found in {mask_shapefile_name}")

    # True outside the shapefile, False inside
    outside_mask = geometry_mask(
        geometries=mask_geoms,
        out_shape=raster_shape,
        transform=transform,
        invert=False
    )

    lidar_to_plot[outside_mask] = np.nan

    print(f"LiDAR masked to: {mask_shapefile_name}")

# --------------------------------------------------
# 6. Work out plot extent
# --------------------------------------------------

if zoom_to_shapefiles and len(shape_layers) > 0:

    if plot_lidar_inside_shapefile_only:
        extent_gdf = shape_layers[mask_shapefile_name]
        xmin, ymin, xmax, ymax = extent_gdf.total_bounds
        xmin -= buffer_m
        ymin -= buffer_m
        xmax += buffer_m
        ymax += buffer_m

    else:
        all_bounds = np.array([
            gdf.total_bounds for gdf in shape_layers.values()
        ])

        xmin = all_bounds[:, 0].min() - buffer_m
        ymin = all_bounds[:, 1].min() - buffer_m
        xmax = all_bounds[:, 2].max() + buffer_m
        ymax = all_bounds[:, 3].max() + buffer_m

else:
    xmin = bounds.left
    ymin = bounds.bottom
    xmax = bounds.right
    ymax = bounds.top

# --------------------------------------------------
# 7. Elevation band settings
# --------------------------------------------------

from matplotlib.colors import BoundaryNorm

# User controls

lowest_elevation = -5      # Lowest colour band
highest_elevation = 60      # Highest colour band
elevation_interval = 5      # Band size

# Create bands automatically

elevation_bins = np.arange(
    lowest_elevation,
    highest_elevation + elevation_interval,
    elevation_interval
)

print("Elevation bins:")
print(elevation_bins)

# Discrete colour bands

norm = BoundaryNorm(
    elevation_bins,
    ncolors=256,
    clip=True
)

# --------------------------------------------------
# 8. Plot LiDAR raster
# --------------------------------------------------

plt.figure(figsize=(12, 10))
ax = plt.gca()

img = ax.imshow(
    lidar_to_plot,
    cmap="terrain",
    norm=norm,
    extent=[
        bounds.left,
        bounds.right,
        bounds.bottom,
        bounds.top
    ],
    origin="upper"
)

# --------------------------------------------------
# 9. Plot shapefiles over the top
# --------------------------------------------------

for name, info in shapefiles.items():

    gdf = shape_layers[name]

    if info["type"] == "polygon":
        gdf.boundary.plot(
            ax=ax,
            color=info["colour"],
            linewidth=info["linewidth"],
            label=name,
            zorder=10
        )

    elif info["type"] == "point":
        gdf.plot(
            ax=ax,
            color=info["colour"],
            markersize=info["markersize"],
            edgecolor="black",
            linewidth=0.5,
            marker="o",
            label=name,
            zorder=20
        )


# --------------------------------------------------
# 10. Finish plot
# --------------------------------------------------

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)

cbar = plt.colorbar(
    img,
    ax=ax,
    boundaries=elevation_bins,
    ticks=elevation_bins
)

cbar.set_label("Elevation (mAOD)")

ax.set_xlabel("Easting")
ax.set_ylabel("Northing")
ax.set_title("Area of Investigation - 10m resampled LiDAR")

if len(shape_layers) > 0:
    ax.legend(
        loc="upper left",
        frameon=True
    )

plt.tight_layout()
plt.show()

In [0]:
import rasterio
import numpy as np
import geopandas as gpd
from rasterio.features import geometry_mask

# --------------------------------------------------
# 1. Set paths
# --------------------------------------------------

lidar_tif = r"/dbfs/FileStore/AndyBeverton/LIDAR_Merged_10m/merged_lidar_DTM_10m.tif"

mask_shp = r"/dbfs/FileStore/AndyBeverton/33035_minus_contributingcatchments_out/33035_minus_contributingcatchments.shp"

# --------------------------------------------------
# 2. User controls
# --------------------------------------------------

thresholds_mAOD = [0, 1, 2, 5, 10]

# --------------------------------------------------
# 3. Open raster
# --------------------------------------------------

with rasterio.open(lidar_tif) as src:
    lidar = src.read(1)
    crs = src.crs
    nodata = src.nodata
    transform = src.transform
    raster_shape = src.shape
    pixel_width = abs(src.transform.a)
    pixel_height = abs(src.transform.e)

pixel_area_m2 = pixel_width * pixel_height

if nodata is not None:
    lidar = np.where(lidar == nodata, np.nan, lidar)

print("Raster CRS:", crs)
print("Pixel size:", pixel_width, "m x", pixel_height, "m")
print("Pixel area:", pixel_area_m2, "m²")

# --------------------------------------------------
# 4. Read mask shapefile
# --------------------------------------------------

mask_gdf = gpd.read_file(mask_shp)

if mask_gdf.crs is None:
    raise ValueError("Mask shapefile has no CRS.")

if mask_gdf.crs != crs:
    mask_gdf = mask_gdf.to_crs(crs)

mask_gdf = mask_gdf[
    mask_gdf.geometry.notnull() &
    ~mask_gdf.geometry.is_empty
].copy()

if len(mask_gdf) == 0:
    raise ValueError("No valid geometries found in mask shapefile.")

# Optional: dissolve into one geometry
mask_gdf["dissolve_id"] = 1
mask_gdf = mask_gdf.dissolve(by="dissolve_id")

polygon_area_m2 = mask_gdf.geometry.area.sum()

print("Polygon area:", polygon_area_m2, "m²")
print("Polygon area:", polygon_area_m2 / 1_000_000, "km²")

# --------------------------------------------------
# 5. Mask raster to polygon
# --------------------------------------------------

mask_geoms = list(mask_gdf.geometry)

outside_mask = geometry_mask(
    geometries=mask_geoms,
    out_shape=raster_shape,
    transform=transform,
    invert=False
)

lidar_masked = lidar.copy()
lidar_masked[outside_mask] = np.nan

valid_elevations = lidar_masked[~np.isnan(lidar_masked)]

if len(valid_elevations) == 0:
    raise ValueError("No valid LiDAR cells found inside the mask polygon.")

# --------------------------------------------------
# 6. Calculate basic elevation statistics
# --------------------------------------------------

valid_cell_count = len(valid_elevations)
valid_raster_area_m2 = valid_cell_count * pixel_area_m2

mean_elev = np.nanmean(valid_elevations)
median_elev = np.nanmedian(valid_elevations)
min_elev = np.nanmin(valid_elevations)
max_elev = np.nanmax(valid_elevations)
std_elev = np.nanstd(valid_elevations)

print("\n--- Elevation statistics inside mask ---")
print(f"Valid raster cells: {valid_cell_count:,}")
print(f"Valid raster area: {valid_raster_area_m2:,.0f} m²")
print(f"Valid raster area: {valid_raster_area_m2 / 1_000_000:,.2f} km²")

print(f"\nMean elevation: {mean_elev:.2f} mAOD")
print(f"Median elevation: {median_elev:.2f} mAOD")
print(f"Minimum elevation: {min_elev:.2f} mAOD")
print(f"Maximum elevation: {max_elev:.2f} mAOD")
print(f"Standard deviation: {std_elev:.2f} m")

# --------------------------------------------------
# 7. Area below selected thresholds
# --------------------------------------------------

print("\n--- Area below elevation thresholds ---")

for threshold in thresholds_mAOD:
    below_count = np.sum(valid_elevations < threshold)
    below_area_m2 = below_count * pixel_area_m2
    below_percent = below_count / valid_cell_count * 100

    print(
        f"Below {threshold} mAOD: "
        f"{below_area_m2:,.0f} m² | "
        f"{below_area_m2 / 1_000_000:,.2f} km² | "
        f"{below_percent:.2f}%"
    )